# Figure 4a & 4b: Neural Clustering Analysis

**Author:** Parviz Ghaderi  
**Date:** 2025  

## Description
This notebook reproduces Figure 4a and 4b from the manuscript:  
_"Contextual gating-capability-scaled-vote response by the frontal cortex supports flexible decision-making"_

The analysis includes:
- **Figure 4a**: Heatmaps showing neural activity patterns across different experimental conditions, along with distributions of brain areas, cell types, cortical layers, and selectivity indices
- **Figure 4b**: Peristimulus time histograms (PSTHs) for each identified neural cluster

## Requirements
```bash
pip install numpy matplotlib scipy
```

## Instructions
1. Ensure the data file `Data_Clustering.pkl` is in the `../processed_data/` directory
2. Run all cells in order
3. Figures will be saved as PDFs in the current directory


## 1. Import Libraries and Set Parameters


In [10]:
# Import required libraries
import os
import pickle
import numpy as np
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.signal import convolve
from scipy.signal.windows import gaussian
import math

matplotlib.use('TkAgg')# Set matplotlib parameters for publication-quality figures
matplotlib.rcParams['pdf.fonttype'] = 42  # Embed fonts as editable text
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.linewidth'] = 0.5
matplotlib.rcParams['xtick.major.width'] = 0.5
matplotlib.rcParams['ytick.major.width'] = 0.5
import matplotlib.pyplot as plt

# Global parameters
BIN_SIZE = 0.01  # 10 ms bin size
GAUSS_WINDOW = 0.1  # 100 ms gaussian window
GAUSSIAN_SMOOTHING = 0  # 0 = no smoothing, 1 = apply smoothing
CMIN, CMAX = -0.2, 0.2  # Colormap limits for heatmaps


## 2. Define Custom Functions


In [11]:
def PG_Scale2Color(value, low_range, high_range, zero):
    """
    Assigns an RGB color to a value by mapping it onto a custom colormap.
    
    Color mapping:
    - Cyan for values below 'low_range'
    - Dark blue through black for negative values approaching 'zero'
    - Black at 'zero'
    - Black through dark red for positive values near 'zero'
    - Yellow for values above 'high_range'
    
    Parameters:
        value (float): The data value to color
        low_range (float): Lower threshold (most negative value)
        high_range (float): Upper threshold (most positive value)
        zero (float): The zero point dividing positive and negative
        
    Returns:
        tuple: RGB values (each between 0 and 1)
    """
    # Define the custom colormap
    colors = [
        (0 / 255, 230 / 255, 230 / 255),   # Cyan for very negative values
        (0.0, 0.1, 1),                     # Dark blue near zero (negative side)
        (0.0, 0.0, 0.0),                   # Black at zero
        (1, 0.1, 0.0),                     # Dark red near zero (positive side)
        (1.0, 1.0, 0.0),                   # Yellow for very positive values
    ]
    
    positions = [0.0, 0.3, 0.5, 0.7, 1.0]
    custom_cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", list(zip(positions, colors)), N=256)
    
    # Normalize the value to the colormap
    if value <= low_range:
        norm_value = 0.0
    elif value >= high_range:
        norm_value = 1.0
    else:
        norm_value = (value - low_range) / (high_range - low_range)
    
    color = custom_cmap(norm_value)
    return color[:3]  # Return RGB only, no alpha


def create_custom_colormap(low_range, high_range, zero, n_colors=256):
    """
    Create a custom colormap using the PG_Scale2Color function.
    
    Parameters:
        low_range (float): Lower threshold
        high_range (float): Upper threshold
        zero (float): Zero point
        n_colors (int): Number of colors in the colormap
        
    Returns:
        tuple: (ListedColormap, Normalize) for use with matplotlib
    """
    colors = np.zeros((n_colors, 3))
    values = np.linspace(low_range, high_range, n_colors)
    for i, val in enumerate(values):
        colors[i, :] = PG_Scale2Color(val, low_range, high_range, zero)
    return mcolors.ListedColormap(colors), mcolors.Normalize(vmin=low_range, vmax=high_range)


def plot_heatmap(ax, psth, cluster_indices, time2plot, start_bin, end_bin, 
                 row_start, row_end, cmin, cmax):
    """
    Plot a heatmap of neural activity for a specific time window.
    
    Parameters:
        ax: Matplotlib axis object
        psth: Neural activity data (neurons x time bins)
        cluster_indices: Indices of neurons in the current cluster
        time2plot: Time vector for x-axis
        start_bin: Starting time bin
        end_bin: Ending time bin
        row_start: Starting row position for this cluster
        row_end: Ending row position for this cluster
        cmin: Minimum value for colormap
        cmax: Maximum value for colormap
    """
    time_segment = time2plot[0:300]
    cdata = psth[cluster_indices, start_bin:end_bin]
    
    # Create custom colormap
    colors = [
        (0 / 255, 230 / 255, 230 / 255),   # Cyan
        (0.0, 0.1, 1),                     # Dark blue
        (0.0, 0.0, 0.0),                   # Black
        (1, 0.1, 0.0),                     # Dark red
        (1.0, 1.0, 0.0),                   # Yellow
    ]
    positions = [0.0, 0.3, 0.5, 0.7, 1.0]
    custom_cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", 
                                                           list(zip(positions, colors)), 
                                                           N=256)
    
    ax.imshow(cdata,
              origin="upper",
              aspect="auto",
              extent=[time_segment[0], time_segment[-1], row_end, row_start],
              vmin=cmin,
              vmax=cmax,
              cmap="coolwarm")
    
    ax.axhline(y=row_end, color="w", linewidth=1)
    ax.set_xlim(0, 3)
    ax.set_ylim(10294, 1)


In [12]:
# Load data from pickle file
current_directory = os.getcwd()
parent_directory = os.path.dirname(current_directory)
pkl_file_path = os.path.join(parent_directory, "processed_data", "Data_Clustering.pkl")

print(f"Loading data from: {pkl_file_path}")
with open(pkl_file_path, "rb") as f:
    Cl = pickle.load(f)




# Extract data arrays
psth = Cl["Neural_Data_normalized_Ordered"]
area = Cl["Areas_Ordered"]
celltype = Cl["Type_Ordered"] 
depth = Cl["Depth_Ordered"]
layer = Cl["Layer_Ordered"]
ccf_location = Cl["CCF_location_Ordered"]
ccf_xyz = Cl["CCF_xyz_Ordered"]
SI = Cl["Selectivity_index"]
cluster_labels_ordered = Cl["Cluster_Counter_Ordered"]


# Define cluster ordering (based on peak response)
cluster_ordered = [1, 24, 23, 2, 16, 5, 14, 11, 21, 7, 9, 29, 20, 28, 25, 17, 13, 22, 6, 26, 27, 10, 3, 19, 15, 12, 18, 8, 4]
unique_clusters = np.unique(cluster_labels_ordered)

# Create time vector
time2plot = np.arange(1, int(15*(1/BIN_SIZE))) / (1/BIN_SIZE)

print(f"Number of neurons: {psth.shape[0]}")
print(f"Number of time bins: {psth.shape[1]}")
print(f"Number of clusters: {len(unique_clusters)}")


Loading data from: d:\Ghaderi_Zenodo_data_code_Final\processed_data\Data_Clustering.pkl
Number of neurons: 10294
Number of time bins: 1500
Number of clusters: 29


## 4. Figure 4a: Heatmaps and Distributions

This figure shows:
- Neural activity heatmaps across 5 experimental conditions
- Area distribution (normalized)
- Cell type distribution
- Cortical layer distribution
- Selectivity index for each cluster


In [13]:
# Create Figure 4a
fig, axs = plt.subplots(nrows=1, ncols=9, figsize=(60/2.54, 25/2.54), sharex=False, sharey=False)

# Initialize row counter
st_cell = 0

# Plot heatmaps for each cluster
for c_id in cluster_ordered:
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    num_neurons_in_cluster = len(cluster_indices)
    
    row_start = st_cell
    row_end = st_cell + num_neurons_in_cluster
    st_cell = row_end
    
    # Plot 5 conditions (3s each)
    for i, ax_idx in enumerate(range(5)):
        plot_heatmap(ax=axs[ax_idx], psth=psth, cluster_indices=cluster_indices, 
                    time2plot=time2plot, start_bin=int(i*3*(1/BIN_SIZE)), 
                    end_bin=int((i+1)*3*(1/BIN_SIZE)), row_start=row_start, 
                    row_end=row_end, cmin=CMIN, cmax=CMAX)
        
        # Format axes
        axs[ax_idx].get_yaxis().set_visible(False)
        axs[ax_idx].set_xlabel("Time (s)")
        axs[ax_idx].set_xticks([0, 1, 2, 3])
        axs[ax_idx].set_xticklabels([-1, 0, 1, 2])
        
        # Remove spines
        for spine in axs[ax_idx].spines.values():
            spine.set_visible(False)
        
        # Add vertical lines at stimulus onset and offset
        axs[ax_idx].axvline(x=1, color='k', linestyle='-', linewidth=0.5)
        axs[ax_idx].axvline(x=2, color='k', linestyle='-', linewidth=0.5)
    
    # Add cluster label
    axs[0].text(-0.5, (row_start + row_end) / 2, f"  {c_id}", color="k")

# Add colorbar for heatmaps
cbar_ax = fig.add_axes([0.05, 0.1, 0.01, 0.08])
fig.colorbar(axs[0].images[-5], cax=cbar_ax, orientation='vertical')

print("Heatmaps completed")


Heatmaps completed


In [14]:
# Plot area distribution (normalized)
unique_areas = [['A1'], ['wS1'], ['wS2'], ['wM2'], ['ALM']]
area_color_map = {
    'A1': 'purple',
    'ALM': 'red',
    'wM2': 'black',
    'wS2': 'green',
    'wS1': 'blue'
}

# Calculate total neurons per area
total_neurons_per_area = {str(a[0]): 0 for a in unique_areas}
for neuron in area:
    area_key = str(neuron[0].item())
    total_neurons_per_area[area_key] += 1

# Plot normalized area distribution for each cluster
for i, c_id in enumerate(cluster_ordered):
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    
    # Calculate area counts
    area_counts = {str(a[0]): 0 for a in unique_areas}
    for neuron_idx in cluster_indices:
        area_key = str(area[neuron_idx][0].item())
        area_counts[area_key] += 1
    
    # Normalize by total neurons per area
    normalized_area_counts = {
        key: count / total_neurons_per_area[key] if total_neurons_per_area[key] > 0 else 0
        for key, count in area_counts.items()
    }
    
    # Convert to percentages
    total_neurons = sum(normalized_area_counts.values())
    area_percentages = [normalized_count / total_neurons * 100 if total_neurons > 0 else 0
                       for normalized_count in normalized_area_counts.values()]
    
    # Calculate row positions
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + len(cluster_indices)
    
    # Plot horizontal stacked bar
    axs[5].barh(y=[(row_start + row_end) / 2], width=area_percentages,
                left=np.cumsum([0] + area_percentages[:-1]),
                color=[area_color_map[str(a[0])] for a in unique_areas],
                height=row_end - row_start)

# Format area distribution subplot
axs[5].set_xlim(0, 100)
axs[5].set_xlabel("Normalized Area Distribution (%)")
axs[5].set_yticks([])
axs[5].legend(
    [plt.Line2D([0], [0], color=color, lw=4) for color in area_color_map.values()],
    [str(a[0]) for a in unique_areas],
    loc='upper right',
    bbox_to_anchor=(5, 1),
    title="Areas"
)
axs[5].set_ylim(0, 10293)
axs[5].invert_yaxis()
axs[5].title.set_text('Area Distribution')

# Add white lines to separate clusters
for i, c_id in enumerate(cluster_ordered):
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + len(np.where(cluster_labels_ordered == c_id)[0])
    axs[5].axhline(y=row_end, color='white', linewidth=1)

print("Area distribution completed")


Area distribution completed


In [15]:
# Plot cell type distribution
unique_celltypes = np.unique(celltype)
celltype_color_map = {str(ct[0]): plt.cm.tab20(i / len(unique_celltypes)) for i, ct in enumerate(unique_celltypes)}
celltype_color_map['FS'] = '#FF69B4'  # Pink for fast-spiking
celltype_color_map['Nan'] = '#808080'  # Gray for unknown
celltype_color_map['RS'] = '#683B00'  # Brown for regular-spiking

# Plot cell type distribution for each cluster
for i, c_id in enumerate(cluster_ordered):
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    num_neurons_in_cluster = len(cluster_indices)
    
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + num_neurons_in_cluster
    
    # Calculate cell type distribution
    # unique_celltypes contains array(['FS']), array(['Nan']), array(['RS'])
    celltype_counts = {str(ct[0].item()): 0 for ct in unique_celltypes}
    for neuron_idx in cluster_indices:
        # celltype has shape (N, 1), so use 2D indexing [neuron_idx, 0] to get array(['RS'])
        celltype_key = str(celltype[neuron_idx, 0].item())
        celltype_counts[celltype_key] += 1
    
    # Convert to percentages
    total_neurons = sum(celltype_counts.values())
    celltype_percentages = [count / total_neurons * 100 for count in celltype_counts.values()]
    
    # Plot horizontal stacked bar
    axs[6].barh(y=[(row_start + row_end) / 2], width=celltype_percentages, 
                left=np.cumsum([0] + celltype_percentages[:-1]),
                color=[celltype_color_map[str(ct[0])] for ct in unique_celltypes], 
                height=row_end - row_start)

# Format cell type subplot
axs[6].set_xlim(0, 100)
axs[6].set_xlabel("Cell Type Distribution (%)")
axs[6].set_yticks([])
axs[6].legend(
    [plt.Line2D([0], [0], color=color, lw=4) for color in celltype_color_map.values()],
    [str(ct[0]) for ct in unique_celltypes],
    loc='upper right',
    bbox_to_anchor=(4, 0.6),
    title="Cell Types"
)
axs[6].title.set_text('Cell Type Distribution')
axs[6].set_ylim(0, 10293)
axs[6].invert_yaxis()

# Add white lines to separate clusters
for i, c_id in enumerate(cluster_ordered):
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + len(np.where(cluster_labels_ordered == c_id)[0])
    axs[6].axhline(y=row_end, color='white', linewidth=1)

print("Cell type distribution completed")


Cell type distribution completed


In [16]:
# Plot layer distribution
# Process layer data
Layer = Cl["Layer_Ordered"]
layer = np.array([str(l.item()) if isinstance(l, np.ndarray) else str(l) for l in Layer], dtype=str)
layer = np.array([l.strip("[]").replace('6a', '6').replace('6b', '6') for l in layer], dtype=str)

unique_layers = np.unique(layer)
print("Unique Layers:", unique_layers)

# Create gradient colormap for layers
base_color = [0.8, 0.8, 0.8]  # Base gray color
layer_color_map = {}
n_layers = len(unique_layers)

for i, l in enumerate(sorted(unique_layers)):
    # Layer 1 is brightest, deeper layers are darker
    gradient_factor = (n_layers - i) / n_layers
    layer_color_map[l] = [c * gradient_factor for c in base_color]

# Plot layer distribution for each cluster
for i, c_id in enumerate(cluster_ordered):
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    num_neurons_in_cluster = len(cluster_indices)
    
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + num_neurons_in_cluster
    
    # Calculate layer distribution
    layer_counts = {str(l): 0 for l in unique_layers}
    for neuron_idx in cluster_indices:
        layer_key = layer[neuron_idx]
        layer_counts[layer_key] += 1
    
    # Convert to percentages
    total_neurons = sum(layer_counts.values())
    layer_percentages = [count / total_neurons * 100 for count in layer_counts.values()]
    
    # Plot horizontal stacked bar
    axs[7].barh(y=[(row_start + row_end) / 2], width=layer_percentages, 
                left=np.cumsum([0] + layer_percentages[:-1]),
                color=[layer_color_map[l] for l in unique_layers], 
                height=row_end - row_start)
    
    # Add cluster label
    axs[7].text(100, (row_start + row_end) / 2, f" {c_id}", va='center', ha='left')

# Format layer subplot
axs[7].set_xlim(0, 100)
axs[7].set_xlabel("Layer Distribution (%)")
axs[7].set_yticks([])
axs[7].legend(
    [plt.Line2D([0], [0], color=color, lw=4) for color in layer_color_map.values()],
    [str(l) for l in unique_layers],
    loc='upper right',
    bbox_to_anchor=(2.95, 0.8),
    title="Layers"
)
axs[7].title.set_text("Layer Distribution")
axs[7].set_ylim(0, 10293)
axs[7].invert_yaxis()

# Add white lines to separate clusters
for i, c_id in enumerate(cluster_ordered):
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + len(np.where(cluster_labels_ordered == c_id)[0])
    axs[7].axhline(y=row_end, color='white', linewidth=1)

print("Layer distribution completed")


Unique Layers: ["'1'" "'2/3'" "'4'" "'5'" "'6'"]
Layer distribution completed


In [17]:
# Plot selectivity index
selectivity_indices = []

# Calculate mean selectivity index for each cluster
for c_id in cluster_ordered:
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    selectivity_index = np.mean(SI[cluster_indices])
    selectivity_indices.append(selectivity_index)
    print(f"Cluster {c_id} mean SI: {selectivity_index:.4f}")

# Plot selectivity index as colored rectangles
for i, c_id in enumerate(cluster_ordered):
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    num_neurons_in_cluster = len(cluster_indices)
    
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + num_neurons_in_cluster
    
    # Get color based on selectivity index
    color = PG_Scale2Color(selectivity_indices[i], -0.5, 0.5, 0)
    axs[8].add_patch(plt.Rectangle((0, row_start), 1, num_neurons_in_cluster, color=color, ec='none'))

# Format selectivity index subplot
axs[8].set_xlim(0, 1)
axs[8].set_xlabel("Selectivity Index")
axs[8].set_yticks([])
axs[8].title.set_text("Selectivity Index")
axs[8].set_ylim(0, 10293)
axs[8].invert_yaxis()

# Add white lines to separate clusters
for i, c_id in enumerate(cluster_ordered):
    row_start = sum(len(np.where(cluster_labels_ordered == prev_cid)[0]) for prev_cid in cluster_ordered[:i])
    row_end = row_start + len(np.where(cluster_labels_ordered == c_id)[0])
    axs[8].axhline(y=row_end, color='white', linewidth=1)

# Add colorbar for selectivity index
cmap, norm = create_custom_colormap(-0.5, 0.5, 0)
cbar_ax = fig.add_axes([0.95, 0.1, 0.015, 0.1])
cbar = matplotlib.colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation='vertical')
cbar.set_label('Selectivity Index', fontsize=12)

# Final formatting
plt.tight_layout(rect=[0, 0, 0.93, 1])
plt.subplots_adjust(wspace=0.02, hspace=0.01, left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.show(block=False)
# Save figure
fig.savefig("../Main_figures_pdf/Ghaderi2025_Figure4_a_heatmaps.pdf", format='pdf', dpi=300, bbox_inches='tight')


Cluster 1 mean SI: 0.0217
Cluster 24 mean SI: 0.0363
Cluster 23 mean SI: -0.0083
Cluster 2 mean SI: 0.0941
Cluster 16 mean SI: 0.0292
Cluster 5 mean SI: 0.0779
Cluster 14 mean SI: 0.0762
Cluster 11 mean SI: 0.5765
Cluster 21 mean SI: 0.1486
Cluster 7 mean SI: -0.0073
Cluster 9 mean SI: 0.0171
Cluster 29 mean SI: 0.0344
Cluster 20 mean SI: 0.0251
Cluster 28 mean SI: 0.2199
Cluster 25 mean SI: 0.2335
Cluster 17 mean SI: 0.1077
Cluster 13 mean SI: 0.0995
Cluster 22 mean SI: 0.0260
Cluster 6 mean SI: 0.0233
Cluster 26 mean SI: -0.0548
Cluster 27 mean SI: -0.0311
Cluster 10 mean SI: -0.0024
Cluster 3 mean SI: -0.0199
Cluster 19 mean SI: -0.0285
Cluster 15 mean SI: -0.0336
Cluster 12 mean SI: -0.0356
Cluster 18 mean SI: -0.1236
Cluster 8 mean SI: -0.0477
Cluster 4 mean SI: -0.1734


C:\Users\crochet\AppData\Local\Temp\ipykernel_8424\158213542.py:44: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.93, 1])
C:\Users\crochet\AppData\Local\Temp\ipykernel_8424\158213542.py:44: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout(rect=[0, 0, 0.93, 1])


## 5. Figure 4b: Clusters [1, 29, 11, 25, 4] PSTH

This figure shows the peristimulus time histograms (PSTHs) for selected clusters, with all 5 experimental conditions overlaid.


In [18]:
# Convert PSTH to firing rate (Hz)
psth_fr = psth / BIN_SIZE

# Define color codes for 5 conditions
colorcodes = np.array([
    [0,   0,   1  ],  # Blue - Condition 1
    [0, 0.5,   1  ],  # Light blue - Condition 2
    [1,   0,   0  ],  # Red - Condition 3
    [1, 0.5,   0  ],  # Orange - Condition 4
    [0,   0,   0  ]   # Black - Condition 5
])

# Select specific clusters to plot
selected_clusters = [1, 29, 11, 25, 4]

# Create figure with 2 rows and 5 columns
fig = plt.figure(figsize=(35/2.54, 15/2.54))

# Create grid layout: 2 rows x 5 columns
gs = fig.add_gridspec(2, 5, height_ratios=[2, 1], width_ratios=[1, 1, 1, 1, 1], 
                     hspace=0.2, wspace=0.3)

# Plot PSTHs in the first row
for i, c_id in enumerate(selected_clusters):
    # Get neurons in this cluster
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    rows_for_cluster = psth_fr[cluster_indices, :]
    
    if rows_for_cluster.size == 0:
        continue
    
    # Calculate mean and SEM
    mean_sig = np.nanmean(rows_for_cluster, axis=0)
    sem_sig = np.nanstd(rows_for_cluster, axis=0) / np.sqrt(rows_for_cluster.shape[0])
    curve_upper, curve_lower = mean_sig + sem_sig, mean_sig - sem_sig
    
    # Create PSTH subplot
    ax_psth = fig.add_subplot(gs[0, i])
    seg_len = int(3 * (1 / BIN_SIZE))  # Length of each segment (300 bins)
    x_segment = time2plot[:seg_len]  # Use the same x range for all conditions
    
    # Plot each condition
    for i_cond in range(5):
        st, en = i_cond * seg_len, (i_cond + 1) * seg_len
        y_mean = mean_sig[st:en]
        y_upper = curve_upper[st:en]
        y_lower = curve_lower[st:en]
        
        # Plot mean ± SEM
        ax_psth.fill_between(x_segment, y_lower, y_upper, color=colorcodes[i_cond], 
                           alpha=0.2, linewidth=0)
        ax_psth.plot(x_segment, y_mean, color=colorcodes[i_cond], linewidth=0.8)
    
    # Add vertical lines at stimulus onset and offset
    ax_psth.axvline(x=1, color='k', linestyle='--', linewidth=1)
    ax_psth.axvline(x=2, color='k', linestyle='--', linewidth=1)
    
    # Format PSTH axes
    ax_psth.set_ylim([-20, 70])
    ax_psth.set_xticks([])
    ax_psth.set_yticks([])
    ax_psth.spines['top'].set_visible(False)
    ax_psth.spines['right'].set_visible(False)
    ax_psth.spines['left'].set_visible(False)
    ax_psth.spines['bottom'].set_visible(False)
    ax_psth.set_title(f"#{c_id}", fontsize=14)

    # Add scale bar to the last PSTH
    ax_psth.plot([2.4, 2.9], [25, 25], color='k', linewidth=2)  # Horizontal scale bar
    ax_psth.plot([2.4, 2.4], [25, 35], color='k', linewidth=2)  # Vertical scale bar
    ax_psth.text(2.65, 24, '0.5s', ha='center', va='top', fontsize=8)
    ax_psth.text(2.3, 30, '10 Hz', ha='right', va='center', fontsize=8)

# Plot distributions in the second row
for i, c_id in enumerate(selected_clusters):
    cluster_indices = np.where(cluster_labels_ordered == c_id)[0]
    
    # Create subplots for distributions (3 subplots per cluster)
    gs_sub = gs[1, i].subgridspec(1, 3, wspace=0.1)
    
    # 1. Layer distribution (vertical stacked bar)
    ax_layer = fig.add_subplot(gs_sub[0, 0])
    
    # Calculate layer distribution for this cluster
    layer_counts = {str(l): 0 for l in unique_layers}
    for neuron_idx in cluster_indices:
        layer_key = layer[neuron_idx]
        layer_counts[layer_key] += 1
    
    # Convert to percentages
    total_neurons = sum(layer_counts.values())
    if total_neurons > 0:
        layer_percentages = [count / total_neurons * 100 for count in layer_counts.values()]
        
        # Plot vertical stacked bar (reverse order so layer 1 is at top)
        ax_layer.bar(x=[0], height=layer_percentages[::-1], 
                     bottom=np.cumsum([0] + layer_percentages[::-1][:-1]),
                     color=[layer_color_map[l] for l in unique_layers[::-1]], 
                     width=0.8)
    
    ax_layer.set_ylim(0, 100)
    ax_layer.set_xticks([])
    ax_layer.set_yticks([])
    ax_layer.set_title('Layer', fontsize=8)
    ax_layer.spines['top'].set_visible(False)
    ax_layer.spines['right'].set_visible(False)
    ax_layer.spines['left'].set_visible(False)
    ax_layer.spines['bottom'].set_visible(False)
    
    # 2. Area distribution (pie chart)
    ax_area = fig.add_subplot(gs_sub[0, 1])
    
    # Calculate area distribution for this cluster
    area_counts = {str(a[0]): 0 for a in unique_areas}
    for neuron_idx in cluster_indices:
        area_key = str(area[neuron_idx][0].item())
        area_counts[area_key] += 1
    
    # Create pie chart
    area_labels = []
    area_sizes = []
    area_colors = []
    for a in unique_areas:
        area_key = str(a[0])
        if area_counts[area_key] > 0:
            area_labels.append(area_key)
            area_sizes.append(area_counts[area_key])
            area_colors.append(area_color_map[area_key])
    
    if area_sizes:
        ax_area.pie(area_sizes, colors=area_colors, autopct='%1.0f%%', 
                   textprops={'fontsize': 6})
    
    ax_area.set_title('Area', fontsize=8)
    
    # 3. Cell type distribution (pie chart)
    ax_celltype = fig.add_subplot(gs_sub[0, 2])
    
    # Calculate cell type distribution for this cluster
    celltype_counts = {str(ct[0].item()): 0 for ct in unique_celltypes}
    for neuron_idx in cluster_indices:
        # celltype has shape (N, 1), so use 2D indexing [neuron_idx, 0] to get array(['RS'])
        celltype_key = str(celltype[neuron_idx, 0].item())
        celltype_counts[celltype_key] += 1
    
    # Create pie chart for FS, RS, and unclassified
    celltype_labels = []
    celltype_sizes = []
    celltype_colors = []
    
    # Check for FS, RS, and unclassified (Nan)
    for ct in ['FS', 'RS', 'Nan']:
        ct_key = str(ct)
        if ct_key in celltype_counts and celltype_counts[ct_key] > 0:
            celltype_labels.append(ct)
            celltype_sizes.append(celltype_counts[ct_key])
            celltype_colors.append(celltype_color_map[ct_key])
    
    if celltype_sizes:
        ax_celltype.pie(celltype_sizes, colors=celltype_colors, autopct='%1.0f%%', 
                       textprops={'fontsize': 6})
    
    ax_celltype.set_title('Cell Type', fontsize=8)
    


# Add legend for PSTH conditions
handles = [mpatches.Patch(color=colorcodes[i]) for i in range(len(colorcodes))]
labels = [f"Condition {i+1}" for i in range(len(colorcodes))]
fig.legend(handles=handles, labels=labels, loc='upper right', fontsize=8, 
          title="Conditions", bbox_to_anchor=(0.98, 0.95))

# Add consolidated legends on the far right
# Layer legend
layer_handles = [plt.Rectangle((0,0),1,1, color=layer_color_map[l]) for l in sorted(unique_layers)]
layer_labels = [str(l) for l in sorted(unique_layers)]
fig.legend(layer_handles, layer_labels, loc='upper right', fontsize=7, 
          title="Layers", bbox_to_anchor=(0.98, 0.75))

# Area legend
area_handles = [plt.Rectangle((0,0),1,1, color=area_color_map[str(a[0])]) for a in unique_areas]
area_labels = [str(a[0]) for a in unique_areas]
fig.legend(area_handles, area_labels, loc='upper right', fontsize=7, 
          title="Areas", bbox_to_anchor=(0.98, 0.55))

# Cell type legend
celltype_handles = [plt.Rectangle((0,0),1,1, color=celltype_color_map[ct]) for ct in ['FS', 'RS', 'Nan']]
celltype_labels = ['FS', 'RS', 'Unclassified']
fig.legend(celltype_handles, celltype_labels, loc='upper right', fontsize=7, 
          title="Cell Types", bbox_to_anchor=(0.98, 0.35))

plt.tight_layout()
plt.show(block=False)
# Save figure
fig.savefig("../Main_figures_pdf/Ghaderi2025_Figure4_b_PSTH_selectedClusters.pdf", format='pdf', dpi=300, bbox_inches='tight')



C:\Users\crochet\AppData\Local\Temp\ipykernel_8424\3187226152.py:192: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



## References

Ghaderi, P. et al. (2025). Contextual gating-capability-scaled-vote response by the frontal cortex supports flexible decision-making.
